# Skript

In [ ]:
from MuellerLib import *
from tifffile import imwrite, imread
from tqdm import tqdm

Proben = np.array([np.eye(4), MuellerLinearPolarizer(0), MuellerLinearPolarizer(45), MuellerRetarder(90,0)])
cmw = imread("data/CalData_Example.tif")
A, W, eps, s = ecm_calibrate(cmw, Proben)

BigSRes = imread("data/SRes_Example.tif")
# BigSRes = np.tile(SRes.reshape(256,256,36), (8, 8, 1)) # Nur weil tif datei zu groß
pictureDim1, pictureDim2 = BigSRes.shape[0], BigSRes.shape[1]
Reconst = np.zeros(((pictureDim1, pictureDim2, 16)))
MMD_LD  = np.zeros((pictureDim1, pictureDim2))
MMD_LD_Ori = np.zeros((pictureDim1, pictureDim2))
MMD_LR = np.zeros((pictureDim1, pictureDim2))
MMD_psi = np.zeros((pictureDim1, pictureDim2))

for i in tqdm(range(pictureDim1)):
	Reconst[i,:,:] = reconstMueller(A, BigSRes[i,:,:], W).reshape(-1, 16)
	MMD_LR[i], MMD_psi[i], MMD_LD[i], MMD_LD_Ori[i] = differentialDecompositionFull(Reconst[i,:,:].reshape(-1, 4, 4))

In [ ]:
plt.figure(figsize=(24, 16))
plt.subplot(2, 2, 1)
plt.title("Original Image")
OrigImage = Probe.reshape(256, 256, 4, 4)
OrigImage = OrigImage.transpose(2, 0, 3, 1)
OrigImage = OrigImage.reshape(4*256, 4*256)
plt.imshow(OrigImage, cmap="viridis")
plt.colorbar()

plt.subplot(2, 2, 2)
plt.title("Reconstructed Image")
RecImage = Reconst.reshape(256, 256, 4, 4)
RecImage = RecImage.transpose(2, 0, 3, 1)
RecImage = RecImage.reshape(4*256, 4*256)
plt.imshow(RecImage, cmap="viridis")
plt.colorbar()

plt.subplot(2,2,3)
plt.title("Reconstruction Error")
Error = OrigImage-RecImage
plt.imshow(Error, cmap="viridis")
plt.colorbar()

plt.show()

plt.figure(figsize=(24, 32))
plt.subplot(3, 2, 1)
plt.title("Diattenuation")
plt.imshow(MMD_LD.reshape(256,256), cmap="viridis")
plt.colorbar()

plt.subplot(3, 2, 2)
plt.title("Depolarization")
plt.imshow(MMD_LD_Ori.reshape(256,256), cmap="viridis")
plt.colorbar()

plt.subplot(3,2,3)
plt.title("Linear Retardance")
plt.imshow(MMD_LR.reshape(256,256), cmap="viridis")
plt.colorbar()

# plt.subplot(3,2,4)
# plt.title("Circular Retardance")
# plt.imshow(MMD_CR.reshape(256,256), cmap="viridis")
# plt.colorbar()

plt.subplot(3,2,5)
plt.title("Orientation")
plt.imshow(MMD_psi.reshape(256,256), cmap="viridis")
plt.colorbar()